# 81 · Streaming — Change Data Capture (Debezium) → Kafka → lakehouse

The companion to notebook `80` (Redpanda), and the **last notebook in the library**. Where `80`
produced and consumed messages on a topic, this one watches a *database* turn its every write
into a stream — **Change Data Capture** — and follows that stream all the way into a live
lakehouse table.

## CDC in one paragraph

**Change Data Capture tails a database's write-ahead log, not its tables.** Every committed
`INSERT`, `UPDATE` and `DELETE` is already recorded in Postgres's WAL for crash recovery;
**Debezium** reads that log through a logical-replication slot and emits one *change event* per
row change onto a Kafka topic. This is **log-tailing, not polling** — there is no `SELECT ... WHERE
updated_at > ?` hammering the source, no missed changes between polls, and deletes are captured
too (a poll of the current table can never see a row that is already gone). The source pays almost
nothing; the change stream is complete and ordered.

## The Debezium change-event envelope

Each event is an envelope describing *one* row change:

| field | meaning |
|-------|---------|
| **`op`** | the operation: `c` create (insert) · `u` update · `d` delete · `r` read (a snapshot row, emitted once when the connector first backfills the table) |
| **`before`** | the row's column values **before** the change (`null` for an insert) |
| **`after`** | the row's column values **after** the change (`null` for a delete) |
| **`source`** | provenance — database, schema, table, LSN, transaction |
| **`ts_ms`** | when the change was captured |

So an update carries *both* the old and new row; a delete carries the old row and a `null`
`after`; the initial snapshot arrives as a run of `op=r` events.

## The end-to-end path (B83)

```
Postgres (musicbrainz.public.cdc_demo)   -- the source table; a write here is a WAL entry
        |  Debezium logical-replication slot (Kafka Connect)
        v
Kafka topic  cdc.musicbrainz.public.cdc_demo   -- one change event per row change
        |  Flink SQL upsert job (cdc_upsert.sql), keyed on id, Iceberg v2 equality-deletes
        v
Iceberg  nessie.datasets_music.cdc_demo_live   -- a LIVE MIRROR of the source, queried via Trino
```

This notebook walks that path in three moves: it shows the **Debezium connector is running**
(section 1), **consumes the change events** and decodes the envelope (section 2), then shows the
**Iceberg mirror** the Flink job maintains from exactly those events (section 3) — closing the
CDC → lakehouse loop.

> **Read-only, throughout.** We consume with a **fresh, throwaway consumer group** (so we read the
> topic from the beginning and commit nothing), and we only `SELECT` from the mirror. Nothing is
> produced to the topic, and neither `cdc_demo` nor `cdc_demo_live` is written. Updating the source
> row `musicbrainz.public.cdc_demo` *would* make Debezium emit a live `u` event — but that is a
> write to the source database, so this notebook does not do it.


## Setup — the CDC client stack

None of these ship in the singleuser base image (which carries `polars`, `pyarrow`, `duckdb`,
`s3fs`, `fastavro`-adjacent tooling): `confluent-kafka` is the Kafka client
(the `[avro,schemaregistry]` extras pull the Schema-Registry client and its HTTP deps), `fastavro`
backs the Confluent-Avro decoder, and `trino` reads the Iceberg mirror. `polars` — used to
render every frame, house style — is already in the image. We reach Kafka Connect's REST API with
the standard library (`urllib`), so no HTTP dependency is needed for that.

In [1]:
%pip install -q "confluent-kafka[avro,schemaregistry]" fastavro trino


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## Connection config — env-driven, in-cluster defaults

Every endpoint is **env-driven**, the same pattern the rest of the wave uses; the committed
defaults are the **in-cluster** service DNS names, so the notebook runs as-is inside the mesh and a
validation run can override any value via env **without editing the notebook**. All four services
are **ClusterIP, no auth** (Redpanda, its Schema Registry, and Kafka Connect are open on the LAN
mesh; the Trino read goes through the `trino-noauth` proxy).

> **The `TRINO_PORT` trap.** Kubernetes injects a `TRINO_PORT=tcp://<ip>:8080` *service* variable
> into any pod in Trino's namespace, so we **never** read `TRINO_PORT` from the environment — the
> port is a literal `8080`. We point at **`trino-noauth`** (a ClusterIP proxy that strips the auth
> header) so reading `iceberg.datasets_music.cdc_demo_live` isn't gated by Ranger. (Notebook `71`
> documents this same proxy and the same literal-port reason.)

In [2]:
import os, json, uuid, time, urllib.request, urllib.error, warnings
from collections import Counter
import polars as pl

# committed defaults = in-cluster DNS; a validation run overrides *_URL / *_BOOTSTRAP via env
REDPANDA_BOOTSTRAP  = os.environ.get("REDPANDA_BOOTSTRAP",  "redpanda.data-mesh.svc.cluster.local:9092")
SCHEMA_REGISTRY_URL = os.environ.get("SCHEMA_REGISTRY_URL", "http://redpanda.data-mesh.svc.cluster.local:8081")
KAFKA_CONNECT_URL   = os.environ.get("KAFKA_CONNECT_URL",   "http://kafka-connect.data-mesh.svc.cluster.local:8083")

# Trino for the Iceberg mirror. Port is a LITERAL 8080 on purpose (never read $TRINO_PORT -- k8s injects a
# colliding tcp://... service var). trino-noauth strips auth, so the read is not gated by Ranger.
TRINO_HOST = os.environ.get("TRINO_HOST", "trino-noauth.data-mesh.svc.cluster.local")
TRINO_PORT = 8080

# The B83 CDC pipeline's fixed coordinates: source table -> Kafka topic -> Iceberg mirror.
CDC_TOPIC      = os.environ.get("CDC_TOPIC", "cdc.musicbrainz.public.cdc_demo")
MIRROR_CATALOG = "iceberg"
MIRROR_SCHEMA  = "datasets_music"
MIRROR_TABLE   = "cdc_demo_live"

def frame(rows, cols):
    """rows + column names -> a polars DataFrame for display (house style, as in 22/40)."""
    return pl.DataFrame(rows, schema=list(cols), orient="row")

print("CDC source topic :", CDC_TOPIC)
print("Kafka Connect    :", KAFKA_CONNECT_URL)
print("Schema Registry  :", SCHEMA_REGISTRY_URL)
print("Iceberg mirror   :", f"{MIRROR_CATALOG}.{MIRROR_SCHEMA}.{MIRROR_TABLE}", "via", TRINO_HOST)

CDC source topic : cdc.musicbrainz.public.cdc_demo
Kafka Connect    : http://kafka-connect.data-mesh.svc.cluster.local:8083
Schema Registry  : http://redpanda.data-mesh.svc.cluster.local:8081
Iceberg mirror   : iceberg.datasets_music.cdc_demo_live via trino-noauth.data-mesh.svc.cluster.local


## 1 · Connector status — is the pipeline live?

Debezium runs as a **Kafka Connect** plugin, so the connector's health is visible through Connect's
REST API. `GET /connectors` lists what's registered; `GET /connectors/<name>` returns a connector's
config; `GET /connectors/<name>/status` returns the runtime state of the connector **and each of
its tasks**. A healthy CDC pipeline shows the connector `RUNNING` and its (single) task `RUNNING` —
that task is the thread actually tailing the WAL and producing to the topic.

We **discover the CDC connector live** from its config (a Debezium Postgres connector whose
`topic.prefix` matches our topic) rather than hard-coding a name.

In [3]:
def connect_get(path):
    with urllib.request.urlopen(KAFKA_CONNECT_URL + path, timeout=15) as r:
        return json.loads(r.read())

connectors = connect_get("/connectors")
print("registered connectors:", connectors)

def is_cdc(name):
    cfg = connect_get(f"/connectors/{name}")["config"]
    klass  = cfg.get("connector.class", "").lower()
    prefix = cfg.get("topic.prefix", "")
    return "postgres" in klass and bool(prefix) and CDC_TOPIC.startswith(prefix)

# resolve the connector feeding our topic; fall back to the sole Postgres connector, then to the first
cdc_name = next((c for c in connectors if is_cdc(c)), None)
if cdc_name is None:
    pg = [c for c in connectors
          if "postgres" in connect_get(f"/connectors/{c}")["config"].get("connector.class", "").lower()]
    cdc_name = pg[0] if pg else connectors[0]
print("CDC connector    :", cdc_name)

registered connectors: ['musicbrainz-cdc']
CDC connector    : musicbrainz-cdc


In [4]:
st = connect_get(f"/connectors/{cdc_name}/status")

rows = [("connector", st["connector"]["state"], st["connector"].get("worker_id", ""))]
rows += [(f"task {t['id']}", t["state"], t.get("worker_id", "")) for t in st["tasks"]]

states = [st["connector"]["state"]] + [t["state"] for t in st["tasks"]]
print(f"connector type   : {st.get('type', '?')}")
print(f"all components RUNNING : {all(s == 'RUNNING' for s in states)}   (states: {states})")
frame(rows, ["component", "state", "worker"])

connector type   : source
all components RUNNING : True   (states: ['RUNNING', 'RUNNING'])


component,state,worker
str,str,str
"""connector""","""RUNNING""","""10.42.0.72:8083"""
"""task 0""","""RUNNING""","""10.42.0.72:8083"""


## 2 · Consume the change events — decode the envelope

Now we read the topic itself. We create a **fresh, random consumer group** with
`auto.offset.reset=earliest` and `enable.auto.commit=false` — so we read from the very first
message and leave no committed offset behind (a true read-only tap). We poll for a few seconds up
to a small cap, then close.

**Detecting the wire format.** The B83 pipeline serializes the topic value as **Confluent-Avro**
(the Flink job reads it as `debezium-avro-confluent` against the Schema Registry). A Confluent-Avro
payload begins with a **magic byte `0x00`** followed by a 4-byte schema id, so we detect it
per-message: byte-0 `0x00` → decode with the registry-backed Avro deserializer; otherwise fall
back to JSON (Debezium's `JsonConverter` output, unwrapping the `{schema, payload}` shape if
present). Either way we recover the same Debezium envelope.

**We assign the partition explicitly and read from its low watermark** — a deterministic bounded
read that doesn't wait on a consumer-group rebalance. We also print the topic's **low/high
watermark offsets** first, because they tell you exactly how much is *retained*: the topic is a
`delete`-policy log with a finite retention window (7 days), so if no write has hit the source
inside that window, the historical snapshot events have aged out and `low == high` (nothing
retained). That is not an error — the connector is still healthy and the Iceberg mirror still holds
the end-state; the change *log* has simply expired. When records are present, each decodes to the
envelope shown below.

In [5]:
import io, contextlib
# confluent-kafka's schema-registry client pulls in authlib, which prints a harmless httpx deprecation
# warning at import; redirect stderr just for these imports so the outputs stay clean.
with contextlib.redirect_stderr(io.StringIO()):
    from confluent_kafka import Consumer, TopicPartition, KafkaError
    from confluent_kafka.schema_registry import SchemaRegistryClient
    from confluent_kafka.schema_registry.avro import AvroDeserializer
    from confluent_kafka.serialization import SerializationContext, MessageField

MAX_EVENTS    = 50     # bounded read cap
POLL_BUDGET_S = 15     # stop after this many seconds regardless

group = f"nb81-readonly-{uuid.uuid4().hex[:8]}"   # FRESH throwaway group; commits nothing
consumer = Consumer({
    "bootstrap.servers": REDPANDA_BOOTSTRAP,
    "group.id": group,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})

# how much is RETAINED? low/high watermark per partition -> retained = high - low.
md = consumer.list_topics(CDC_TOPIC, timeout=10).topics[CDC_TOPIC]
lows, retained = {}, 0
for p in md.partitions:
    lo, hi = consumer.get_watermark_offsets(TopicPartition(CDC_TOPIC, p), timeout=10)
    lows[p] = lo
    retained += hi - lo
    print(f"partition {p}: low={lo} high={hi} retained={hi - lo}")
print(f"total retained on topic : {retained}")

# assign explicitly at each partition's low watermark (deterministic bounded read, no rebalance wait,
# and no out-of-range reset from seeking below what's retained)
consumer.assign([TopicPartition(CDC_TOPIC, p, lo) for p, lo in lows.items()])

sr = SchemaRegistryClient({"url": SCHEMA_REGISTRY_URL})
avro_des = AvroDeserializer(sr)   # resolves the writer schema from the registry by embedded id

def decode_value(raw):
    """Debezium value -> (envelope, wire_format). Confluent-Avro (magic 0x00) or JSON, detected per-message."""
    if raw is None:
        return None, "tombstone"
    if raw[0] == 0:                                   # Confluent wire format: 0x00 + schema id + Avro
        return avro_des(raw, SerializationContext(CDC_TOPIC, MessageField.VALUE)), "avro"
    env = json.loads(raw)
    return env.get("payload", env), "json"            # JsonConverter w/ schemas.enable wraps in {schema,payload}

events, wire_fmt = [], None
deadline = time.time() + POLL_BUDGET_S
while len(events) < min(MAX_EVENTS, retained) and time.time() < deadline:
    msg = consumer.poll(1.0)
    if msg is None or msg.error():                    # None = no message this tick; error = transient/EOF
        continue
    env, fmt = decode_value(msg.value())
    wire_fmt = wire_fmt or fmt
    if env is not None:
        events.append(env)

consumer.close()
print(f"\nconsumed {len(events)} change events from {CDC_TOPIC}")
print(f"wire format      : {wire_fmt if wire_fmt else 'n/a (nothing retained to decode)'}")

partition 0: low=12 high=13 retained=1
total retained on topic : 1



consumed 1 change events from cdc.musicbrainz.public.cdc_demo
wire format      : avro


In [6]:
OP_MEANING = {"r": "read (snapshot)", "c": "create (insert)", "u": "update", "d": "delete"}

def op_row(env):
    before = env.get("before") or {}
    after  = env.get("after") or {}
    op     = env.get("op")
    rid    = after.get("id", before.get("id"))
    return (op, OP_MEANING.get(op, op), rid, before.get("note"), after.get("note"), env.get("ts_ms"))

print("op distribution  :", dict(Counter(e.get("op") for e in events)))
frame([op_row(e) for e in events], ["op", "meaning", "id", "before.note", "after.note", "ts_ms"])

op distribution  : {'u': 1}


op,meaning,id,before.note,after.note,ts_ms
str,str,i64,str,str,i64
"""u""","""update""",11,"""bravo-v2""","""bravo-v3""",1788368173412


In [7]:
# one full decoded envelope -- the before/after/op/ts_ms structure, verbatim
sample = events[0] if events else None
if sample:
    print("envelope keys    :", list(sample.keys()))
    print(json.dumps({k: sample.get(k) for k in ("op", "before", "after", "ts_ms")}, indent=2, default=str))
else:
    print("no change events on the topic yet (nothing to decode)")

envelope keys    : ['before', 'after', 'source', 'transaction', 'op', 'ts_ms', 'ts_us', 'ts_ns']
{
  "op": "u",
  "before": {
    "id": 11,
    "name": null,
    "note": "bravo-v2",
    "updated_at": "2026-07-15T14:30:27.591348Z"
  },
  "after": {
    "id": 11,
    "name": null,
    "note": "bravo-v3",
    "updated_at": "2026-07-15T14:30:27.591348Z"
  },
  "ts_ms": 1788368173412
}


**What each `op` means here.** The topic opens with a run of **`op=r`** events — Debezium's
initial *snapshot* of `cdc_demo`, one read event per existing row, so a fresh consumer can rebuild
the whole table before any live change arrives. After the snapshot, live writes to the source show
up as **`c`** (a new row — `before` null, `after` the inserted row), **`u`** (a change — `before`
the old row, `after` the new one), or **`d`** (a removal — `before` the old row, `after` null). If
the frame above is all `r`, the source table simply hasn't been written to since the snapshot —
a healthy, quiet CDC stream. And if the watermark line above showed `low == high` (nothing
retained), the change log's 7-day retention window has passed with no recent write, so even the
snapshot events have aged out: there is nothing to decode, yet the connector is `RUNNING` and the
mirror (section 3) still holds the end-state those events produced. Updating
`musicbrainz.public.cdc_demo` would append a live `u` event here — we don't, since that writes the
source.

## 3 · The Iceberg mirror — the materialized end-state

The change stream is the *log*; the **Iceberg table `iceberg.datasets_music.cdc_demo_live` is the
materialized end-state**. A Flink SQL job (`cdc_upsert.sql`) consumes the same topic and **upserts
by `id`** into an Iceberg v2 table (equality-deletes on the primary key), so the table always
mirrors the source row-for-row — an insert adds a row, an update replaces one, a delete removes
one. It is *not* an append log of changes; it is the live picture the changes add up to.

We read it through Trino (`trino-noauth`, literal port `8080`), and prove the connection by the
version it reports back rather than by echoing the address.

In [8]:
import trino

conn = trino.dbapi.connect(host=TRINO_HOST, port=TRINO_PORT, user="nb81",
                           catalog=MIRROR_CATALOG, schema=MIRROR_SCHEMA, http_scheme="http")

def tq(sql):
    cur = conn.cursor()
    cur.execute(sql)
    return frame(cur.fetchall(), [d[0] for d in cur.description])

ver = tq("SELECT version() AS v").item(0, "v")
print(f"connected (Trino {ver}) via {TRINO_HOST} -- endpoint from env")

mirror = tq(f"SELECT id, note FROM {MIRROR_TABLE} ORDER BY id")
print(f"{MIRROR_CATALOG}.{MIRROR_SCHEMA}.{MIRROR_TABLE}: {mirror.height} rows (the live mirror)")
mirror

connected (Trino 468) via trino-noauth.data-mesh.svc.cluster.local -- endpoint from env


iceberg.datasets_music.cdc_demo_live: 2 rows (the live mirror)


id,note
i64,str
10,"""alpha"""
11,"""bravo-v3"""


**Tie the stream to the mirror.** The mirror is what you get by *folding* the change stream:
apply each event by `id` — `r`/`c`/`u` set the row to its `after` value, `d` removes it — and the
result is the current table. Below we fold the events we consumed and line the end-state up against
what Trino returns from the Iceberg table. We consumed only what is *currently retained* on the
topic, so the fold matches the mirror when the retained prefix is the snapshot plus recent changes;
if the log has aged out (nothing retained), the fold is empty while the mirror still shows the
durable end-state — the Flink job did this same fold continuously over *all* events, back when they
had not yet expired, and Iceberg persists the result independent of topic retention.

In [9]:
# fold the consumed change stream to its end-state, then compare to the Iceberg mirror
state = {}
for e in events:
    op     = e.get("op")
    after  = e.get("after") or {}
    before = e.get("before") or {}
    rid    = after.get("id", before.get("id"))
    if op == "d":
        state.pop(rid, None)
    elif after:
        state[rid] = after.get("note")

folded = frame(sorted(state.items()), ["id", "note"])
print(f"stream folded to end-state : {folded.height} rows")
print(f"Iceberg mirror (Trino)     : {mirror.height} rows")
folded

stream folded to end-state : 1 rows
Iceberg mirror (Trino)     : 2 rows


id,note
i64,str
11,"""bravo-v3"""


## Close — when to reach for CDC, and the library complete

**Reach for Change Data Capture when you need to react to *every* database write without touching
the write path.** Because it tails the log rather than polling tables, it is the right tool for:

| use case | why CDC fits |
|----------|--------------|
| **Replication / migration** | stream one database's changes into another with no dual-writes and no downtime |
| **Cache / search invalidation** | invalidate or re-index exactly the rows that changed, the moment they change |
| **Event-driven architecture** | turn plain table writes into a domain event stream other services consume — no app changes |
| **Lakehouse mirroring** | keep an analytics table (here, Iceberg) continuously in sync with an operational source — *this notebook's path* |

The trade-off is operational: a logical-replication slot on the source, a Connect worker to run, a
schema to manage — worth it precisely when polling would be lossy, heavy, or blind to deletes.

**The streaming layer is complete.** With `80` (Redpanda: produce/consume, schemas, consumer
groups) and this notebook (`81`: CDC → Kafka → Flink → Iceberg), the library now covers streaming
end to end — from a raw topic to a database's change log landing as a live lakehouse table.

**And with it, the whole B81 library is complete.** The library now spans the full platform:
**formats** (`01`–`04`) → **storage & versioning** (`10`–`11`) → **query & federation** (`20`–`22`)
→ **vector & graph** (`30`–`33`) → **transform & semantic** (`40`–`41`) → **feature & ML**
(`50`–`51`) → **AI & RAG** (`60`–`62`) → **governance & quality** (`70`–`72`) → **streaming**
(`80`–`81`). Every notebook runs end-to-end against the live mesh — that is the test. This is the
last one.